In [9]:
import numpy as np
from torchsummary import summary
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

def preprocess(train_file, test_file):
    train = np.load(train_file)
    test = np.load(test_file)

    X_train, y_train = train["X"], train["y"]
    X_test, y_test = test["X"], test["y"]

    X_train = torch.tensor(X_train, dtype=torch.float32)
    X_test = torch.tensor(X_test, dtype=torch.float32)

    X_mean, X_std = X_train.mean(dim=0), X_train.std(dim=0)

    X_train = (X_train - X_mean) / X_std
    X_test = (X_test - X_mean) / X_std

    K = (X_train.T @ X_train) / (X_train.shape[0] - 1)
    D, V = torch.linalg.eigh(K)

    X_mean = X_train.mean(dim=0)
    D = D.flip(0)
    V = V.flip(1)

    X_train = (X_train - X_mean) @ V[:, :32] @ torch.diag(D[:32] ** -0.5)
    X_test = (X_test - X_mean) @ V[:, :32] @ torch.diag(D[:32] ** -0.5)


    return (X_train, y_train), (X_test, y_test)

(X_train, y_train), (X_test, y_test) = preprocess("train1.npz", "test1.npz")

In [10]:
class Model(nn.Module):
    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(32, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(p=0.5),

            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.Tanh(),
            nn.Dropout(p=0.5),

            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(p=0.5),

            nn.Linear(32, 10)
        )

    def forward(self, x):
        return self.net(x)

model = Model()
summary(model, (32,))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Linear-1                  [-1, 128]           4,224
       BatchNorm1d-2                  [-1, 128]             256
              ReLU-3                  [-1, 128]               0
           Dropout-4                  [-1, 128]               0
            Linear-5                   [-1, 64]           8,256
       BatchNorm1d-6                   [-1, 64]             128
              Tanh-7                   [-1, 64]               0
           Dropout-8                   [-1, 64]               0
            Linear-9                   [-1, 32]           2,080
      BatchNorm1d-10                   [-1, 32]              64
             ReLU-11                   [-1, 32]               0
          Dropout-12                   [-1, 32]               0
           Linear-13                   [-1, 10]             330
Total params: 15,338
Trainable params: 

In [11]:
def train(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct = 0, 0

    for X, y in loader:
        X, y = X.to(device), y.to(device)

        optimizer.zero_grad()
        outputs = model(X)
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * len(y)
        correct += (outputs.argmax(dim=1) == y).sum().item()

    return total_loss / len(loader.dataset), correct / len(loader.dataset)


def test(model, loader, criterion, device):
    model.eval()
    total_loss, correct = 0, 0

    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)

            outputs = model(X)
            loss = criterion(outputs, y)

            total_loss += loss.item() * len(y)
            correct += (outputs.argmax(dim=1) == y).sum().item()

    return total_loss / len(loader.dataset), correct / len(loader.dataset)

train_dataset = TensorDataset(
    torch.tensor(X_train, dtype=torch.float32),
    torch.tensor(y_train, dtype=torch.long)
)
test_dataset = TensorDataset(
    torch.tensor(X_test, dtype=torch.float32),
    torch.tensor(y_test, dtype=torch.long)
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = Model().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

epochs = 100

for epoch in range(epochs):
    train_loss, train_acc = train(model, train_loader, criterion, optimizer, device)
    test_loss, test_acc = test(model, test_loader, criterion, device)

    print(
        f"Epoch {epoch + 1}/{epochs} | "
        f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2%} | "
        f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.2%}"
    )

torch.save(model.state_dict(), "model1.pth")

/var/folders/by/8fc6d5ln63xg5yzt1yyss0jm0000gn/T/ipykernel_83733/3085161326.py:37: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(X_train, dtype=torch.float32),
/var/folders/by/8fc6d5ln63xg5yzt1yyss0jm0000gn/T/ipykernel_83733/3085161326.py:41: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(X_test, dtype=torch.float32),


Epoch 1/100 | Train Loss: 2.2584 | Train Acc: 17.12% | Test Loss: 1.9068 | Test Acc: 59.25%
Epoch 2/100 | Train Loss: 1.9665 | Train Acc: 33.94% | Test Loss: 1.5698 | Test Acc: 85.25%
Epoch 3/100 | Train Loss: 1.7039 | Train Acc: 49.19% | Test Loss: 1.2561 | Test Acc: 88.75%
Epoch 4/100 | Train Loss: 1.4934 | Train Acc: 56.56% | Test Loss: 0.9970 | Test Acc: 91.00%
Epoch 5/100 | Train Loss: 1.2558 | Train Acc: 66.56% | Test Loss: 0.7936 | Test Acc: 93.50%
Epoch 6/100 | Train Loss: 1.1312 | Train Acc: 67.69% | Test Loss: 0.6374 | Test Acc: 94.50%
Epoch 7/100 | Train Loss: 1.0199 | Train Acc: 71.06% | Test Loss: 0.5025 | Test Acc: 95.00%
Epoch 8/100 | Train Loss: 0.8981 | Train Acc: 75.81% | Test Loss: 0.4040 | Test Acc: 94.25%
Epoch 9/100 | Train Loss: 0.8034 | Train Acc: 78.12% | Test Loss: 0.3524 | Test Acc: 95.00%
Epoch 10/100 | Train Loss: 0.7729 | Train Acc: 77.06% | Test Loss: 0.3057 | Test Acc: 95.50%
Epoch 11/100 | Train Loss: 0.7331 | Train Acc: 79.94% | Test Loss: 0.2553 | Tes